In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv('train.csv')
X = df.drop(columns=["id", "is_fraud"])
y = df["is_fraud"]

cat_variables = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_variables = X.select_dtypes(include=["number"]).columns.tolist()
X = df.drop(columns=["id", "is_fraud"])
y = df["is_fraud"].astype(int)

# Cast categoricals so LightGBM uses native handling
for c in cat_variables:
    X[c] = X[c].astype("category")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(X))
val_aucs = []
models = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMClassifier(
        n_estimators=2000,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=50,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=(y_tr == 0).sum() / (y_tr == 1).sum(),
        objective="binary",
        random_state=42,
        n_jobs=-1,
    )
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    oof[va_idx] = model.predict_proba(X_va)[:, 1]
    auc = roc_auc_score(y_va, oof[va_idx])
    val_aucs.append(auc)
    models.append(model)
    print(f"Fold {fold}: AUC = {auc:.5f}, best_iter = {model.best_iteration_}")

print(f"\nMean fold AUC: {np.mean(val_aucs):.5f} ± {np.std(val_aucs):.5f}")
print(f"Overall OOF AUC: {roc_auc_score(y, oof):.5f}")

[LightGBM] [Info] Number of positive: 9352, number of negative: 550616
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014598 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1745
[LightGBM] [Info] Number of data points in the train set: 559968, number of used features: 24
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.016701 -> initscore=-4.075447
[LightGBM] [Info] Start training from score -4.075447
Fold 0: AUC = 0.88234, best_iter = 1
[LightGBM] [Info] Number of positive: 9351, number of negative: 550617
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.016526 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1745
[LightGBM] [Info] Number of data points in the train set: 559968, number 

In [3]:
test = pd.read_csv("test.csv")
test_ids = test["id"]
X_test = test.drop(columns=["id"])
for c in cat_variables:
    X_test[c] = X_test[c].astype("category")

test_proba = np.mean([m.predict_proba(X_test)[:, 1] for m in models], axis=0)
pd.DataFrame({"id": test_ids, "is_fraud": test_proba}).to_csv("submission2.csv", index=False)